# IOL-AI 2026 — a submission preview on Linguini (Colab T4)

This notebook is a **hands-on preview of what a competition submission actually does**. We run a candidate model on real [Linguini](https://huggingface.co/datasets/facebook/linguini) linguistics-olympiad problems — the same format as the [IOL-AI 2026](https://iolai.org) test set — so you can see the *raw model output*, how it gets parsed into answers, and how those answers are scored against the references.

**Why a Colab T4?** Competition submissions run on a T4 (16 GB VRAM) with a 30-minute limit. Testing on the same GPU gives representative memory usage and runtimes. Set **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

**How this notebook is organised** — one step per cell, so each does a single clear thing:

0. **Setup** — install dependencies, silence noisy logs
1. **Load Linguini** and pick 8 problems
2. **Load the model**
3. **Prompt and parser** — how we ask, how we read the answer back
4. **One problem, end to end** — see a full raw generation
5. **Generate** answers for all 8 problems *(no scoring yet)*
6. **Inspect** a raw generation
7. **Score** the predictions against the references
8. **Summary** and projection against the 30-minute budget

**Data note:** please do not re-host Linguini problems in plaintext (contamination risk) — load them from the Hub only.

## 0. Setup

Setup takes ~3 minutes. `gptqmodel` is required to load AWQ models with recent `transformers`. The install writes its output to a log file so the cell output stays clean.

In [ ]:
# Install quietly — full output goes to the log file so nothing noisy shows here.
# If it fails, we print the tail of the log instead of failing silently.
!pip install -q -U transformers accelerate datasets sacrebleu bitsandbytes gptqmodel > /tmp/pip_install.log 2>&1 && echo "dependencies installed (full log: /tmp/pip_install.log)" || { echo "INSTALL FAILED — last 25 log lines:"; tail -25 /tmp/pip_install.log; }

In [ ]:
# Imports, quiet logging, and a one-line GPU check.
# Colab renders anything sent to stderr on a pink background, so we mute library chatter here.
import os, logging, warnings

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"   # no download progress bars on stderr

warnings.filterwarnings("ignore")
logging.getLogger().setLevel(logging.ERROR)         # drop INFO/WARNING chatter globally
for name in ("huggingface_hub", "transformers", "datasets", "accelerate",
             "torchao", "gptqmodel", "bitsandbytes"):
    logging.getLogger(name).setLevel(logging.ERROR)

import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    # Fail fast: the AWQ model needs a GPU, so don't waste a run on CPU.
    raise RuntimeError("No GPU detected. Set Runtime -> Change runtime type -> T4 GPU, then Run all.")
props = torch.cuda.get_device_properties(0)
print(f"GPU: {torch.cuda.get_device_name(0)} | {props.total_memory / 1e9:.0f} GB")

## 1. Load Linguini and pick 8 problems

Each row is one complete problem: `context` (the language data), `query` (the instruction and numbered items), `answer` (the reference answers, sometimes with several accepted alternatives per item).

In [ ]:
from datasets import load_dataset
import pandas as pd, ast

ds = load_dataset("facebook/linguini")
df = ds["test"].to_pandas()
print("problems:", len(df), "| task types:", list(df.task_type.unique()))

def gold_list(x):
    """Parse the answer field into a list of items, each a list of accepted alternatives."""
    if isinstance(x, (list, tuple)):
        v = list(x)
    else:
        try:
            v = ast.literal_eval(str(x).strip())
            if not isinstance(v, list):
                v = [v]
        except Exception:
            v = [str(x).strip()]
    return [[str(a).strip() for a in it] if isinstance(it, (list, tuple)) else [str(it).strip()] for it in v]

N_PROBLEMS = 8
sample = df.sample(N_PROBLEMS, random_state=0)   # fixed seed: every model sees the same problems
rows = [{"context": str(r["context"]), "query": str(r["query"]),
         "task_type": r["task_type"], "gold": gold_list(r["answer"])}
        for _, r in sample.iterrows()]
print(f"{len(rows)} problems ready | task types in sample:", [r["task_type"] for r in rows])

In [ ]:
# What one problem looks like.
r = rows[0]
print("CONTEXT (first 600 chars):\n", r["context"][:600])
print("\nQUERY:\n", r["query"][:400])
print("\nGOLD (first 3 items):", r["gold"][:3])

## 2. Load the model

Current pick: **Qwen2.5-14B-Instruct-AWQ** (non-thinking, greedy). A 14B fits the T4 only quantized: fp16 needs ~2 GB per billion parameters (28 GB, too big); 4-bit needs ~0.6 GB (about 10 GB, fits).

Thinking models (Qwen3-14B, DeepSeek-R1 distills) were tested and ruled out: they generate thousands of reasoning tokens per problem on a T4 and exceed the 30-minute limit.

**Platform note:** the competition sandbox cannot load AWQ/GPTQ models (no network at run time, required library not preinstalled). For an actual submission, ship full-precision weights and load them in 4-bit with bitsandbytes instead: uncomment the `quantization_config` variant below and use `Qwen/Qwen2.5-14B-Instruct`.

In [ ]:
import time, io, contextlib
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_REPO = "Qwen/Qwen2.5-14B-Instruct-AWQ"
MAX_NEW_TOKENS = 1536          # this model answers in a few hundred tokens; raise for thinking models
GEN_KWARGS = dict(do_sample=False)   # greedy: deterministic, fine for non-thinking models

kw = dict(device_map="auto")
# Platform-compatible alternative (full-precision repo + 4-bit at load time):
# MODEL_REPO = "Qwen/Qwen2.5-14B-Instruct"
# kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True,
#     bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)

torch.cuda.reset_peak_memory_stats()
t0 = time.time()
with contextlib.redirect_stderr(io.StringIO()):   # keep the AWQ loader's chatter out of the cell
    tok = AutoTokenizer.from_pretrained(MODEL_REPO)
    model = AutoModelForCausalLM.from_pretrained(MODEL_REPO, **kw).eval()
load_s = time.time() - t0
print(f"loaded {MODEL_REPO}")
print(f"{load_s:.0f}s to load | peak VRAM {torch.cuda.max_memory_allocated() / 1e9:.1f} GB / 16 GB")

## 3. The prompt and the answer parser

The model is asked to reason, then output `FINAL ANSWERS:` followed by numbered lines. Numbered lines (rather than brackets) avoid collisions with IPA notation, where square brackets are part of the answer itself, and let us align answers by item number even if the model skips one. The parser maps each expected item number to its line and returns exactly one answer per item.

In [ ]:
import re

SYSTEM = (
    "You solve International Linguistics Olympiad problems by reasoning from the data given. "
    "You may face a task type you have never seen: read the instruction and adapt. "
    "For fill-in-the-blank tables, check which column each blank is in: if the blank is in the "
    "meaning/translation column, answer with the meaning; if it is in the language column, answer "
    "with the word form in that language. For matching tasks answer with the option letter only. "
    "For number tasks answer with digits or the written-out number, as asked. "
    "Reason step by step first. Then write a line saying exactly FINAL ANSWERS: followed by one "
    "line per item, formatted as the item number, a period, and the bare answer, using the same "
    "item numbers as the problem. Example:\nFINAL ANSWERS:\n1. first answer\n2. second answer\n"
    "Do not add brackets, quotes, or any other text on those lines."
)

def expected_items(query):
    """Item numbers announced in the query, e.g. (1)...(10) or '17. ...'."""
    nums = set(int(n) for n in re.findall(r"\((\d{1,3})\)", query))
    for ln in query.splitlines():
        m = re.match(r"^\s*(\d{1,3})[.)]\s", ln)
        if m:
            nums.add(int(m.group(1)))
    return sorted(nums)

def strip_think(t):
    """Drop the <think> block of reasoning models before parsing."""
    return t.split("</think>")[-1] if "</think>" in t else t

def extract(text, item_nums):
    """Pull one bare answer per expected item from the model's FINAL ANSWERS block."""
    text = strip_think(text)
    m = list(re.finditer(r"(?im)^[\s>*#-]*final answers?\s*[:.]?\s*$", text))
    if m:
        text = text[m[-1].end():]
    numbered = {}
    for ln in text.splitlines():
        mm = re.match(r"^\s*(\d{1,3})[.)]\s*(.+?)\s*$", ln)
        if mm:
            numbered.setdefault(int(mm.group(1)), mm.group(2).strip())
    if numbered and item_nums:
        return [numbered.get(k, "") for k in item_nums]
    if numbered and not item_nums:
        return [numbered[k] for k in sorted(numbered)]
    out = []
    for ln in text.splitlines():
        ln = re.sub(r"^[\s>*#-]+", "", ln)
        ln = re.sub(r"^\d{1,3}[.)]\s*", "", ln).strip()
        if ln:
            out.append(ln)
    return out

def fmt_gold(alts):
    """Render a gold item (its accepted alternatives) as one string, to line up with a prediction."""
    if not alts:
        return ""
    return alts[0] + (f"  (or: {', '.join(alts[1:])})" if len(alts) > 1 else "")

In [ ]:
def solve(row):
    """Generate the model's answer for one problem. Returns (preds, raw_text, n_tokens, seconds)."""
    items = expected_items(row["query"])
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": row["context"].strip() + "\n\n" + row["query"].strip()}]
    try:
        enc = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                      return_tensors="pt", return_dict=True)
    except TypeError:
        enc = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt")
    enc = enc.to(model.device)
    t1 = time.time()
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, **GEN_KWARGS)
    dt = time.time() - t1
    n_tok = out.shape[-1] - enc["input_ids"].shape[-1]
    raw = tok.decode(out[0][enc["input_ids"].shape[-1]:], skip_special_tokens=True)
    preds = extract(raw, items)
    n = len(row["gold"])
    preds = (preds + [""] * n)[:n]   # exactly one prediction per item, aligned to the gold items
    return preds, raw, n_tok, dt

## 4. One problem, end to end

The most instructive view: the model's full raw output, then what the parser extracted next to the references. The gap between raw and parsed is score lost to formatting, not intelligence.

In [ ]:
preds, raw, n_tok, dt = solve(rows[0])
print(f"{n_tok} tokens in {dt:.0f}s ({n_tok / dt:.1f} tok/s)\n")
print("===== RAW OUTPUT =====")
print(raw)
print("\n===== PARSED PREDICTION vs REFERENCE =====")
for k, (p, alts) in enumerate(zip(preds, rows[0]["gold"]), 1):
    hit = any(p.strip().lower() == a.lower() for a in alts)
    mark = "OK " if hit else "-  "
    print(f"  {mark} item {k}: pred: {p!r:<22} gold: {fmt_gold(alts)!r}")

## 5. Generate answers for all 8 problems

This is the **generate** step — the model runs here and nowhere else. Each problem's raw output and parsed prediction is stored in `generations`, so the later cells can inspect and score them **without re-running the model**.

In [ ]:
generations = []
t_run = time.time()
for i, row in enumerate(rows):
    try:
        preds, raw, n_tok, dt = solve(row)
        status = f"{n_tok:>4} tok / {dt:.0f}s"
    except Exception as e:                      # keep the run alive so finished problems aren't lost
        preds = [""] * len(row["gold"])
        raw, n_tok, dt = f"[generation failed: {e}]", 0, 0.0
        status = f"FAILED: {e}"
        torch.cuda.empty_cache()
    generations.append(dict(problem=i + 1, task=row["task_type"],
                            preds=preds, gold=row["gold"],
                            raw=raw, tokens=n_tok, seconds=dt))
    print(f"[{i + 1}/{len(rows)}] {row['task_type']:<14} {status}")
print(f"\ngenerated {len(generations)} answers in {(time.time() - t_run) / 60:.1f} min")

## 6. Inspect a raw generation

Read exactly what the model produced for any problem. Change the index to look at a different one (0–7).

In [ ]:
g = generations[4]   # change the index to inspect a different problem
print(f"Problem {g['problem']} — {g['task']}  ({g['tokens']} tokens, {g['seconds']:.0f}s)\n")
print(g["raw"])

## 7. Score the predictions against the references

Scoring mirrors the competition: **exact match** (case-insensitive; correct if the prediction matches any accepted alternative) and **chrF** (character n-gram partial credit, best alternative). The official score is `100 * sqrt(weighted_EM * weighted_chrF)`, so both matter. This scores the stored `generations` — the model does not run again.

In [ ]:
from sacrebleu.metrics import CHRF
chrf = CHRF()

def score_problem(preds, gold):
    """Exact-match and chrF for one problem, averaged over its items."""
    n = len(gold)
    if n == 0:
        return 0.0, 0.0
    em = sum(any(p.strip().lower() == a.lower() for a in alts) for p, alts in zip(preds, gold)) / n
    cf = sum(max((chrf.sentence_score(p, [a]).score for a in alts), default=0.0) for p, alts in zip(preds, gold)) / n
    return em, cf

results = []
for g in generations:
    em, cf = score_problem(g["preds"], g["gold"])
    results.append(dict(problem=g["problem"], task=g["task"], items=len(g["gold"]),
                        em=round(em, 2), chrf=round(cf, 1),
                        tokens=g["tokens"], seconds=round(g["seconds"], 1)))
res = pd.DataFrame(results)
res

In [ ]:
# Predictions next to references, in matching format — this is where formatting costs show up.
for g in generations:
    em, _ = score_problem(g["preds"], g["gold"])
    print(f"Problem {g['problem']} — {g['task']:<14} EM {em:.2f}")
    for k, (p, alts) in enumerate(zip(g["preds"], g["gold"]), 1):
        hit = any(p.strip().lower() == a.lower() for a in alts)
        mark = "OK " if hit else "-  "
        print(f"  {mark} item {k}: pred: {p!r:<24} gold: {fmt_gold(alts)!r}")
    print()

## 8. Summary and projection against the budget

Average scores, throughput, and a projection of total runtime against the competition's 30-minute limit (the real test set is about 6 problems).

In [ ]:
import math

em_avg, cf_avg = res.em.mean(), res.chrf.mean()
proxy = 100 * math.sqrt(max(em_avg, 0) * max(cf_avg / 100, 0))
proj_min = (load_s + 120 + 6 * res.seconds.mean()) / 60   # load + overhead + ~6 problems

speed = (res.tokens / res.seconds.where(res.seconds > 0)).mean()   # skip any failed (0s) problems
print(f"model: {MODEL_REPO}")
print(f"avg EM {em_avg:.2f} | avg chrF {cf_avg:.1f} | proxy score {proxy:.1f}")
print(f"avg speed {speed:.1f} tok/s | avg {res.seconds.mean():.0f}s per problem")
print(f"projected full-set runtime: {proj_min:.1f} min / 30 min budget ->",
      "OK" if proj_min < 27 else "TOO SLOW")

## Findings so far (updated 2026-07-08)

Qwen2.5-1.5B runs everywhere but does not reason: it copies vocabulary from the context regardless of the prompt. Thinking models (Qwen3-14B-AWQ, DeepSeek-R1 distills) exceed the 30-minute limit on a T4: thousands of reasoning tokens per problem, cut mid-thought. Qwen2.5-14B-Instruct-AWQ is the current pick: about 10 GB VRAM, 30–60 s per problem, and it induces morphological rules on Linguini instead of copying. For actual submissions the platform cannot load AWQ/GPTQ (offline sandbox, library not preinstalled): ship full-precision weights and load in 4-bit with bitsandbytes, as in the commented variant of section 2.